# Transformations and Actions

Spark has two kinds of DataFrame operations:

| | Transformations | Actions |
|---|---|---|
| What they do | Describe a new DataFrame built from an existing one | Ask Spark for a result |
| When they run | **Lazily**: nothing runs yet, Spark only adds a step to the plan | **Right away**: they start a Spark job |
| Return | A new DataFrame | A value, or output on screen or disk |
| Examples | `select`, `where`/`filter`, `groupBy`, `orderBy`, `limit` | `show`, `count`, `collect`, `take`, `write`, `display` |

DataFrames are **immutable**. A transformation never changes the original DataFrame. It always returns a new one.

## 1. Top 3 zip codes with the most calls

**Question:** which 3 zip codes account for the most fire calls?

**Approach:** group the calls by `Zipcode`, count the rows in each group, sort from highest to lowest, and keep the first 3.

`%sql` on the first line makes this cell run as SQL instead of Python. In a SQL cell, comments start with `--`, not `#`.

In [ ]:
%sql
-- Columns to return: call type, zip code, and the number of calls in each group
select callType, Zipcode, count(*) as total_calls
-- Source table (catalog.schema.table)
from dev.spark_db.sf_fire_calls
-- Ignore rows where the zip code is missing
where Zipcode is not null
-- One group per (call type, zip code) pair
group by CallType, Zipcode
-- Biggest counts first
order by total_calls desc
-- Keep only the top 3
limit 3

## 2. The same query with the DataFrame API

The SQL query above can be broken into 3 parts:

1. **Read** the source table into a DataFrame
2. **Transform** it: select, filter, group and count, sort, limit (these are all lazy)
3. **Act** on it: show the result (this is what actually runs the job)

Each SQL clause maps to a DataFrame method:

| SQL | DataFrame API |
|---|---|
| `from table` | `spark.read.table("...")` |
| `select cols` | `.select("col1", "col2")` |
| `where cond` | `.where("cond")` or `.filter(...)` |
| `group by ... count(*)` | `.groupBy(...).count()` |
| `order by ... desc` | `.orderBy("col", ascending=False)` |
| `limit n` | `.limit(n)` |

### Step 1: read the data

In [ ]:
# Get a SparkSession. Here it connects to Databricks through Databricks Connect
# (see the startup banner in the output).
from pyspark.shell import spark

# Load the table as a DataFrame. Only the schema is read here, not the data.
fire_df = spark.read.table("dev.spark_db.sf_fire_calls")

# Outside a Databricks notebook, display() prints only the schema, like
# DataFrame[...] below. Use fire_df.show(5) to see actual rows.
display(fire_df)

### Step 2: transformations

Each line creates a new DataFrame from the one before. **No data is processed yet.** Spark only builds up a plan of steps, so this cell finishes immediately.

In [ ]:
# SELECT callType, Zipcode: keep only the 2 columns we need
df_1 = fire_df.select("callType", "Zipcode")

# WHERE: drop rows with a missing value
# Note: the SQL version filters on Zipcode, but this filters on callType
df_2 = df_1.where("callType is not null")

# GROUP BY + COUNT(*): one row per (callType, Zipcode) pair, with a "count" column
df_3 = df_2.groupBy("callType", "Zipcode").count()

# ORDER BY count DESC: biggest counts first
df_4 = df_3.orderBy("count", ascending=False)

# LIMIT 3: keep the top 3 rows
df_5 = df_4.limit(3)

### Step 3: action

`.show()` is an **action**. Calling it makes Spark optimize the whole plan from Step 2 and run it as a job, then print the result. This is the first time any data is actually read.

In [ ]:
# Action: runs the job and prints the top 3 rows as a text table
df_5.show()

### Key takeaways

- Steps 1 and 2 return almost instantly because transformations are lazy and only build a plan.
- The real work happens only when an action like `.show()` is called.
- Spark looks at the whole chain at once, so it can optimize it. For example, it reads only the 2 selected columns and stops early because of `limit(3)`.
- The SQL and DataFrame versions produce the same kind of plan, so neither one is faster.
- You can also chain the steps instead of naming each DataFrame:

```python
(fire_df.select("callType", "Zipcode")
        .where("Zipcode is not null")
        .groupBy("callType", "Zipcode").count()
        .orderBy("count", ascending=False)
        .limit(3)
        .show())
```